# Thí nghiệm H-FraudGT — Phần 2: Đo riêng R/F/M và cổng độ tin cậy

Phần này tách riêng **H-R, H-F và H-M** để đo đóng góp của từng nhóm đặc trưng, đồng thời chạy **HG** để kiểm nghiệm cơ chế giảm độ tin cậy khi lịch sử quan sát còn ít.

> **Mục đích.** Kết quả của phần này sẽ được ghép với baseline A ở Phần 1; vì vậy cột chênh lệch so với A có thể để trống trong CSV tạm thời.

| Thiết lập | Giá trị |
|---|---|
| Dữ liệu | AML Small-HI |
| Seed | 43 |
| GPU | 2 × NVIDIA T4 |
| Chọn mô hình | Validation F1 tại threshold 0.50 |
| Thời gian dự kiến | 4,5–5 giờ |

Toàn bộ mô hình dùng cùng sampler, số epoch, loss và siêu tham số. Tập test
không được dùng để chọn epoch. Hãy chạy notebook bằng **Save Version → Save &
Run All** để Kaggle thực thi từ một môi trường sạch và giữ lại output.

**Luồng thực nghiệm:** kiểm tra môi trường → xác nhận dữ liệu → kiểm tra chống
rò rỉ → huấn luyện → chọn epoch bằng validation → đánh giá test → đóng gói.


In [ ]:
# Thiết lập duy nhất cần kiểm tra trước khi Run All
from pathlib import Path

REPO_URL = 'https://github.com/mhiunguyen/TH-FraudGT.git'
REPO = Path('/kaggle/working/TH-FraudGT')
MODELS = ['H-R', 'H-F', 'H-M', 'HG']
GPU_INDICES = [0, 1]
RUN_TRAINING = True

EVIDENCE = Path('/kaggle/working/final_history_evidence')
EVIDENCE.mkdir(parents=True, exist_ok=True)
print('Models:', MODELS)
print('GPU indices:', GPU_INDICES)
print('RUN_TRAINING:', RUN_TRAINING)

## 1. Xác nhận tài nguyên thực nghiệm

Cell tiếp theo in phiên bản Python, PyTorch, CUDA và thông tin của hai GPU.
Giữ output này làm bằng chứng về môi trường chạy; nếu đưa vào báo cáo, chỉ cần
chụp phần tên GPU, VRAM và phiên bản PyTorch/CUDA.


In [ ]:
import platform, subprocess, sys
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {prop.name}; VRAM={prop.total_memory / 1024**3:.2f} GiB')
subprocess.run(['nvidia-smi'], check=False)

## 2. Cố định phiên bản mã nguồn

Repository được clone trực tiếp từ GitHub và commit hash được ghi lại. Commit
hash giúp xác định chính xác phiên bản code đã tạo ra kết quả, kể cả khi mã
nguồn tiếp tục được chỉnh sửa sau này.


In [ ]:
import os, subprocess

os.chdir('/kaggle/working')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

os.chdir(REPO)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
required = [
    REPO / 'scripts/run_final_history_experiments.py',
    REPO / 'scripts/summarize_final_history_experiments.py',
    REPO / 'scripts/package_final_history_evidence.py',
    REPO / 'configs/AML-Small-HI/AML-Small-HI-History-Final-Seed43.yaml',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError('Repository chưa có bộ chạy cuối: ' + str(missing))
print('Repository:', REPO)
print('Commit:', commit)

## 3. Chuẩn bị thư viện tương thích với GPU Kaggle

Các gói PyTorch Geometric phải khớp với phiên bản PyTorch và CUDA của session.
Cell này tự tạo đúng địa chỉ wheel rồi kiểm tra lại các import quan trọng. Nếu
cell chưa in đủ phiên bản thư viện thì không tiếp tục huấn luyện.


In [ ]:
import subprocess, sys, torch

torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('PyG wheel index:', wheel_url)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url,
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(REPO / 'requirements-kaggle.txt'),
], check=True)

import torch_geometric, torch_sparse, torch_scatter, yaml
print('torch_geometric:', torch_geometric.__version__)
print('torch_sparse:', torch_sparse.__version__)
print('torch_scatter:', torch_scatter.__version__)
print('PyYAML:', yaml.__version__)

## 4. Xác nhận dữ liệu AML Small-HI

Notebook tìm `HI-Small_Trans.csv` trong Kaggle Input, đưa file về đúng vị trí
mà FraudGT sử dụng, sau đó ghi số giao dịch, kích thước và SHA-256. Manifest
này chứng minh các lần chạy dùng cùng một bản dữ liệu.


In [ ]:
import hashlib, json, shutil

candidates = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if not candidates:
    raise FileNotFoundError(
        'Không tìm thấy HI-Small_Trans.csv. Hãy Add Input bộ IBM AML rồi chạy lại cell.'
    )
source = candidates[0]
destination = REPO / 'data/AML/HI-Small_Trans.csv'
destination.parent.mkdir(parents=True, exist_ok=True)
if not destination.exists() or destination.stat().st_size != source.stat().st_size:
    shutil.copy2(source, destination)

digest = hashlib.sha256()
with destination.open('rb') as stream:
    for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
        digest.update(block)
with destination.open('rb') as stream:
    rows = max(sum(1 for _ in stream) - 1, 0)
manifest = {
    'source': str(source), 'destination': str(destination),
    'bytes': destination.stat().st_size, 'transaction_rows': rows,
    'sha256': digest.hexdigest(),
}
(EVIDENCE / 'dataset_manifest.json').write_text(
    json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

## 5. Kiểm tra tính hợp lệ trước khi huấn luyện

Trước khi dùng GPU, notebook kiểm tra cú pháp, chạy unit test cho đặc trưng
lịch sử và xác nhận giao dịch tại thời điểm `t` không nhìn thấy giao dịch cùng
hoặc sau `t`. Bảng cấu hình sau đó phải cho thấy mọi mô hình cùng seed, epoch,
checkpoint và tiêu chí `f1_t50`; chỉ nhóm R/F/M hoặc reliability được phép khác.


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'compileall', '-q', 'fraudGT', 'scripts'], check=True)
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_history_features.py', 'tests/test_threshold_selection.py',
], check=True)
subprocess.run([
    sys.executable, 'scripts/run_final_history_experiments.py',
    '--repo', str(REPO), '--prepare-only',
], check=True)
print('Kiểm tra logic và sinh config: OK')

In [ ]:
import pandas as pd, yaml

rows = []
cfg_dir = REPO / 'generated_configs_final_history'
for model in MODELS:
    path = cfg_dir / f'AML-Small-HI-Final-{model}-Seed43.yaml'
    cfg = yaml.safe_load(path.read_text(encoding='utf-8'))
    rows.append({
        'model': model,
        'history': cfg['dataset']['add_history'],
        'groups': '+'.join(cfg['dataset']['history_groups']),
        'reliability': cfg['dataset']['history_reliability'],
        'best_metric': cfg['metric_best'],
        'checkpoint': cfg['train']['enable_ckpt'],
        'epochs': cfg['optim']['max_epoch'],
    })
config_table = pd.DataFrame(rows)
display(config_table)
assert set(config_table['best_metric']) == {'f1_t50'}
assert config_table['checkpoint'].all()

## 6. Huấn luyện các cấu hình của phần này

Các mô hình được phân thành từng cặp để hai T4 hoạt động đồng thời. Dòng
`heartbeat` cho biết tiến trình vẫn sống dù cell không in log từng epoch. GPU
thấp trong lúc lấy mẫu hoặc tạo cache là bình thường. Thời gian dự kiến của
phần này là **4,5–5 giờ**.


In [ ]:
import subprocess, sys, time

if RUN_TRAINING:
    command = [
        sys.executable, '-u', 'scripts/run_final_history_experiments.py',
        '--repo', str(REPO), '--gpus', *map(str, GPU_INDICES),
        '--models', *MODELS,
    ]
    print('Command:', ' '.join(command))
    started = time.time()
    subprocess.run(command, cwd=REPO, check=True)
    print(f'Hoàn tất sau {(time.time() - started) / 3600:.2f} giờ')
else:
    print('Bỏ qua huấn luyện theo RUN_TRAINING=False')

## 7. Chọn epoch mà không nhìn vào tập test

Với mỗi mô hình, epoch tốt nhất được chọn bằng F1 trên validation tại threshold
0.50. Precision, recall, F1 và AUC trên test chỉ được đọc tại đúng epoch đó.
Cell cũng đối chiếu epoch trong `best.ckpt`; nếu `checkpoint_ok` không phải
`True`, chưa được sử dụng kết quả trong báo cáo.


In [ ]:
summary_path = EVIDENCE / 'summary_final_history_seed43.csv'
subprocess.run([
    sys.executable, 'scripts/summarize_final_history_experiments.py',
    '--results-root', str(REPO / 'results_final_history'),
    '--output', str(summary_path), '--models', *MODELS,
], cwd=REPO, check=True)
results = pd.read_csv(summary_path)
display(results[[
    'model', 'best_epoch_by_validation', 'val_f1', 'test_f1',
    'delta_f1_vs_A', 'test_precision', 'test_recall', 'test_auc',
    'checkpoint_ok',
]])
assert results['checkpoint_ok'].all()

## 8. So sánh F1 giữa các cấu hình

Biểu đồ dùng cùng một thang đo và ghi trực tiếp F1 (%) trên từng cột. Đây là
hình nên lưu để trình bày ablation; bảng CSV vẫn là nguồn số liệu chính khi
viết báo cáo.


In [ ]:
import matplotlib.pyplot as plt

plot = results.set_index('model').loc[MODELS].reset_index()
fig, ax = plt.subplots(figsize=(10, 5.2))
colors = [
    '#777777' if model == 'A' else '#222222' if model == 'HG' else '#356a9a'
    for model in plot['model']
]
bars = ax.bar(plot['model'], plot['test_f1'] * 100, color=colors)
if 'A' in set(plot['model']):
    baseline = float(plot.loc[plot['model'] == 'A', 'test_f1'].iloc[0]) * 100
    ax.axhline(baseline, color='black', linewidth=1, linestyle='--',
               label=f'FraudGT gốc: {baseline:.2f}%')
for bar, value in zip(bars, plot['test_f1'] * 100):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.6,
            f'{value:.2f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('F1 trên tập kiểm thử (%)')
ax.set_xlabel('Cấu hình')
ax.set_title('Ablation đặc trưng lịch sử và reliability gate — seed 43')
ax.grid(axis='y', alpha=0.2)
if 'A' in set(plot['model']):
    ax.legend()
fig.tight_layout()
plot_path = EVIDENCE / 'final_history_f1_seed43.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)

## 9. Đóng gói khả năng tái lập

Ngoài kết quả, đồ án cần giữ được code, config, môi trường, log và trọng số.
Hai cell sau ghi commit, `pip freeze`, thông tin runtime rồi tạo một ZIP có
manifest SHA-256 cho toàn bộ bằng chứng.


In [ ]:
import json, platform, subprocess, sys

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
status = subprocess.check_output(['git', 'status', '--short'], cwd=REPO, text=True)
(EVIDENCE / 'source_manifest.json').write_text(json.dumps({
    'repository': REPO_URL, 'commit': commit, 'git_status': status,
}, indent=2), encoding='utf-8')
(EVIDENCE / 'pip_freeze.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True),
    encoding='utf-8')
(EVIDENCE / 'runtime.txt').write_text(
    f'Python: {sys.version}\nPlatform: {platform.platform()}\n'
    f'PyTorch: {torch.__version__}\nCUDA runtime: {torch.version.cuda}\n',
    encoding='utf-8')
print('Commit:', commit)
print('Git status sau run:\n' + (status or '(clean)'))

In [ ]:
output_base = Path('/kaggle/working/H_FraudGT_Final_Part2')
subprocess.run([
    sys.executable, 'scripts/package_final_history_evidence.py',
    '--repo', str(REPO),
    '--results-root', str(REPO / 'results_final_history'),
    '--summary', str(summary_path),
    '--logs-dir', '/kaggle/working/final_history_logs',
    '--evidence-dir', str(EVIDENCE),
    '--output-base', str(output_base),
], cwd=REPO, check=True)
archive = output_base.with_suffix('.zip')
print('TẢI FILE NÀY VỀ MÁY:', archive)
print('Kích thước:', archive.stat().st_size / 1024**2, 'MiB')

## 10. Kiểm tra trước khi rời Kaggle

Chỉ xem phần này hoàn tất khi đã kiểm tra đủ các mục sau:

- [ ] Bảng summary hiển thị đủ các mô hình của phần này.
- [ ] Tất cả dòng `checkpoint_ok` đều là `True`.
- [ ] Biểu đồ F1 đã được tạo và không thiếu cột.
- [ ] Mỗi run có `best.ckpt`, checkpoint phục hồi và ba file `stats.json`.
- [ ] Đã tải **`/kaggle/working/H_FraudGT_Final_Part2.zip`** về máy.
- [ ] Phiên bản notebook trên Kaggle có trạng thái `Successful`.

Nếu batch run thất bại, đọc phần cuối log để xác định model dừng ở đâu. Trong
cùng một Draft Session, runner có thể tiếp tục từ checkpoint; nếu Kaggle đã
xóa toàn bộ session thì cần chạy lại phần chưa có ZIP.
